# 🦌 Doñana — 4. ¿Mejora el detector con aumentación de datos? (Albumentations)

Entrenamos el **mismo modelo con los mismos hiperparámetros** varias veces, cambiando solo cómo se aumenta el
**conjunto de entrenamiento**, y comparamos todos los modelos en el **mismo test**.

Adaptación del notebook de aumentación de clase:
- Las transformaciones se definen con `albumentations.Compose` + `BboxParams(format="pascal_voc")`, igual que allí,
  para que las cajas se transformen junto con la imagen (y se descarten si un recorte deja fuera al animal).
- En vez de escribir las imágenes aumentadas a disco (`run_data_augmentation` + XML), se generan **al vuelo**:
  `num_augments = 2` significa que cada foto original aparece 1 vez tal cual + 2 veces aumentada (dataset ×3), y las
  versiones aumentadas son **distintas en cada época**. Mismo efecto de "aumentar los datos", sin ocupar disco.
- La API de Albumentations 2.x cambió algunos nombres respecto al notebook original
  (`ShiftScaleRotate` → `Affine`, `CoarseDropout(max_holes=…)` → `CoarseDropout(num_holes_range=…)`, etc.).
- **Valid y test nunca se aumentan.**

### Experimentos
| Nombre | Qué prueba |
|---|---|
| `baseline` | Sin aumentación. Referencia. |
| `baseline_repetido` | Control: cada foto repetida 3 veces **sin** transformar. Mismas iteraciones que los experimentos ×3; si una aumentación mejora sobre este control, la mejora viene de la aumentación y no de entrenar más pasos. |
| `geometrica` | Volteo horizontal, traslación/escala/rotación pequeña, recorte que conserva el animal. |
| `fotometrica` | Brillo, contraste, gamma, tono/saturación, CLAHE. |
| `camara_trampa` | Específicas del dominio: escala de grises (imita IR nocturno), ruido de sensor, desenfoque de movimiento, compresión JPEG, niebla, sombras. |
| `oclusion` | Huecos aleatorios (vegetación que tapa al animal), como el `dirty_aug` original. |
| `combinada` | Todo lo anterior con probabilidades moderadas. |
| `combinada_balanceada` | `combinada`, pero el nº de copias depende de la clase: las clases raras (tejón, `Otros`…) se aumentan mucho y los cérvidos nada. Ataca el desbalanceo. |

> ⏱️ **Coste:** cada experimento es un entrenamiento completo. Con Faster R-CNN ResNet50 a 1024 px en una T4, el dataset ×3
> tarda ~25-30 min por época. Para comparar muchas aumentaciones conviene `faster_rcnn_v3` (MobileNet, ~4× más rápido) y/o
> `image_size = 800`, y después repetir solo la mejor aumentación con el modelo grande. Los resultados se guardan en Drive
> y los experimentos ya terminados se saltan: puedes ejecutar el notebook en varias sesiones.

In [ ]:
!pip install -q -U "albumentations>=2.0.5" torchmetrics pycocotools

In [ ]:
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/donana")  # donana_data.py, donana_det.py y processed/ están aquí
    LOCAL_DIR = Path("/content")                         # disco local de Colab (rápido)
else:
    PROJECT_DIR = Path.cwd()
    LOCAL_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR))

PROCESSED_DIR = PROJECT_DIR / "processed"            # anotaciones, splits y caché comprimida
MODELS_DIR = PROJECT_DIR / "models"                  # checkpoints (en Drive, sobreviven a desconexiones)
IMAGES_DIR = LOCAL_DIR / "dona_24_1024"              # imágenes reducidas (lado mayor 1024 px)
CACHE_ZIP = PROCESSED_DIR / "dona_24_1024.zip"

import json
import numpy as np
import pandas as pd
import torch
import donana_data as dd
import donana_det as det

dd.ensure_image_cache(IMAGES_DIR, CACHE_ZIP)
annotations = pd.read_csv(PROCESSED_DIR / "annotations.csv")
CLASS_NAMES = json.loads((PROCESSED_DIR / "dataset_info.json").read_text())["class_names"]
NUM_CLASSES = len(CLASS_NAMES) + 1   # + fondo (clase 0)
train_df = annotations[annotations["split"] == "train"]
valid_df = annotations[annotations["split"] == "valid"]
test_df = annotations[annotations["split"] == "test"]
print(f"Clases ({NUM_CLASSES - 1}): {CLASS_NAMES}")
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")

import albumentations as A
print("albumentations", A.__version__)

## 1. Visualizar cada aumentación
Antes de entrenar, comprobamos que las transformaciones tienen sentido y que las cajas siguen al animal (equivale a `test_augmentor`).

In [ ]:
preview_set = det.DonanaDataset(train_df, IMAGES_DIR)
day_idx = int(np.flatnonzero(train_df.groupby("filePath", sort=False)["lighting"].first().values == "día")[0]) if (train_df["lighting"] == "día").any() else 0

for name in det.AUGMENTATIONS:
    det.show_augmentations(preview_set, det.get_augmentation(name), CLASS_NAMES, record_idx=day_idx, n=4, title=name)

### Definir tu propia aumentación
Puedes añadir pipelines al diccionario `det.AUGMENTATIONS` (una función que devuelve la lista de transformaciones) y
usarlos en `EXPERIMENTS`. Lista de transformaciones: https://albumentations.ai/docs/reference/supported-targets-by-transform/
(solo valen las que soportan `bboxes`).

In [ ]:
det.AUGMENTATIONS["mi_aumentacion"] = lambda: [
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.ToGray(p=0.3),
]
det.show_augmentations(preview_set, det.get_augmentation("mi_aumentacion"), CLASS_NAMES, record_idx=day_idx, title="mi_aumentacion")

## 2. Configuración de los experimentos
Todos los experimentos comparten modelo, hiperparámetros y semilla; solo cambia la aumentación.

In [ ]:
seed = 0
model_name = "faster_rcnn_v3"        # cambia a "faster_rcnn_v1" para el modelo de 02_train (más lento)
pretrained = True
image_size = 1024
use_gpu, use_amp, num_workers = True, True, 2
num_epochs = 12
early_stopping_patience = 4
batch_size = 4
lr, lr_momentum, lr_decay = 0.005, 0.9, 1e-4
lr_factor, lr_patience, lr_threshold, lr_min = 0.1, 2, 1e-3, 1e-5
max_grad_norm = 0.0

NUM_AUGMENTS = 2                     # copias aumentadas por imagen -> dataset x3

EXPERIMENTS = [
    {"name": "baseline",             "augmentation": None,            "num_augments": 0},
    {"name": "baseline_repetido",    "augmentation": None,            "num_augments": NUM_AUGMENTS},
    {"name": "geometrica",           "augmentation": "geometrica",    "num_augments": NUM_AUGMENTS},
    {"name": "fotometrica",          "augmentation": "fotometrica",   "num_augments": NUM_AUGMENTS},
    {"name": "camara_trampa",        "augmentation": "camara_trampa", "num_augments": NUM_AUGMENTS},
    {"name": "oclusion",             "augmentation": "oclusion",      "num_augments": NUM_AUGMENTS},
    {"name": "combinada",            "augmentation": "combinada",     "num_augments": NUM_AUGMENTS},
    {"name": "combinada_balanceada", "augmentation": "combinada",     "balance_target": 800, "max_copies": 10},
]

EXPERIMENTS_DIR = MODELS_DIR / f"augmentation_{model_name}_{image_size}"
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
device = det.get_device(use_gpu)

# Valid y test: siempre sin aumentar
valid_loader = det.make_loader(det.DonanaDataset(valid_df, IMAGES_DIR), batch_size, shuffle=False, num_workers=num_workers)
test_set = det.DonanaDataset(test_df, IMAGES_DIR)
test_loader = det.make_loader(test_set, batch_size, shuffle=False, num_workers=num_workers)
lighting = test_df.groupby("filePath", sort=False)["lighting"].first().values
subsets = {"día": lighting == "día", "noche": lighting == "noche (IR)"}

# Tamaño del train en cada experimento
def build_train_set(exp):
    transform = det.get_augmentation(exp["augmentation"])
    copies = det.balanced_copies(train_df, exp["balance_target"], exp.get("max_copies", 10)) if "balance_target" in exp else None
    return det.DonanaDataset(train_df, IMAGES_DIR, transform=transform, num_augments=exp.get("num_augments", 0), copies=copies)

pd.DataFrame([{"experimento": e["name"], "imágenes de train por época": len(build_train_set(e))} for e in EXPERIMENTS])

Copias por clase en `combinada_balanceada`:

In [ ]:
copies = det.balanced_copies(train_df, 800, 10)
per_image = train_df.groupby("filePath", sort=False)["label_name"].first()
pd.DataFrame({"clase": per_image.values, "copias": copies}).groupby("clase").agg(
    originales=("copias", "size"), copias_por_imagen=("copias", "first")
).assign(total=lambda d: d["originales"] * (1 + d["copias_por_imagen"])).loc[CLASS_NAMES]

## 3. Entrenar y evaluar cada experimento
Cada experimento guarda `best.pt`, `last.pt`, `history.json` y `test_results.json` en su carpeta. Si `test_results.json` ya existe, se salta.

In [ ]:
def run_experiment(exp):
    out_dir = EXPERIMENTS_DIR / exp["name"]
    results_path = out_dir / "test_results.json"
    if results_path.exists():
        print(f"[{exp['name']}] ya hecho, se carga {results_path}")
        return json.loads(results_path.read_text())

    print(f"\n========== {exp['name']} ==========")
    det.set_seed(seed)
    train_set = build_train_set(exp)
    train_loader = det.make_loader(train_set, batch_size, shuffle=True, num_workers=num_workers, seed=seed)

    model = det.torchvision_model(model_name, NUM_CLASSES, pretrained=pretrained, image_size=image_size).to(device)
    optimizer = det.build_optimizer(model, lr=lr, momentum=lr_momentum, weight_decay=lr_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=lr_factor, patience=lr_patience, threshold=lr_threshold, min_lr=lr_min
    )
    config = dict(model_name=model_name, num_classes=NUM_CLASSES, image_size=image_size, class_names=CLASS_NAMES,
                  seed=seed, batch_size=batch_size, lr=lr, experiment=exp)
    history = det.fit(model, optimizer, scheduler, train_loader, valid_loader, device, num_epochs, out_dir, config,
                      early_stopping_patience=early_stopping_patience, use_amp=use_amp, max_grad_norm=max_grad_norm)

    best_model, _ = det.load_trained_model(out_dir / "best.pt", device)
    res = det.evaluate(best_model, test_loader, device, CLASS_NAMES, subsets=subsets)
    summary = {
        "experimento": exp["name"],
        "imágenes train/época": len(train_set),
        "mejor época": history["best_epoch"],
        "épocas": history["current_epoch"],
        "mAP valid": history["best_value"],
        **det.summary_row(res),
        "ap_por_clase": res["ap_por_clase"],
    }
    results_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    del model, best_model, optimizer
    torch.cuda.empty_cache()
    return summary

all_results = [run_experiment(exp) for exp in EXPERIMENTS]

## 4. Comparación en test

In [ ]:
table = pd.DataFrame([{k: v for k, v in r.items() if k != "ap_por_clase"} for r in all_results]).set_index("experimento")
baseline_map = table.loc["baseline", "mAP"]
table["Δ mAP vs baseline"] = table["mAP"] - baseline_map
metric_cols = ["mAP", "Δ mAP vs baseline", "mAP@50", "mAP@75", "mAP agnóstico", "Localización", "Acierto especie",
               "map_día", "map_noche", "mAP valid", "mejor época", "imágenes train/época"]
table[[c for c in metric_cols if c in table.columns]].round(4).style.background_gradient(
    subset=["mAP", "mAP@50", "Acierto especie"], cmap="Greens")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
order = table["mAP"].sort_values().index
colors = ["#888888" if n.startswith("baseline") else "#2a7ab9" for n in order]
axes[0].barh(order, table.loc[order, "mAP"], color=colors)
axes[0].axvline(baseline_map, color="black", linestyle="--", linewidth=1, label="baseline")
axes[0].set_xlabel("mAP@[.5:.95] en test")
axes[0].set_xlim(table["mAP"].min() * 0.9, table["mAP"].max() * 1.03)
axes[0].legend()

if "map_día" in table.columns:
    x = np.arange(len(order))
    axes[1].barh(x - 0.2, table.loc[order, "map_día"], height=0.4, label="día", color="#f0a030")
    axes[1].barh(x + 0.2, table.loc[order, "map_noche"], height=0.4, label="noche (IR)", color="#404070")
    axes[1].set_yticks(x, order)
    axes[1].set_xlabel("mAP@[.5:.95] en test")
    axes[1].legend()
plt.tight_layout()
plt.savefig(EXPERIMENTS_DIR / "comparacion.png", dpi=120)
plt.show()

### AP por clase
¿Qué aumentación ayuda a qué especie? Lo esperable es que el balanceo ayude sobre todo a las clases raras y que `camara_trampa` ayude de noche.

In [ ]:
ap_table = pd.DataFrame({r["experimento"]: r["ap_por_clase"] for r in all_results}).T[CLASS_NAMES]
ap_delta = ap_table - ap_table.loc["baseline"]

fig, ax = plt.subplots(figsize=(1.2 * len(CLASS_NAMES) + 3, 0.55 * len(ap_delta) + 2))
lim = np.nanmax(np.abs(ap_delta.values)) or 0.01
im = ax.imshow(ap_delta.values, cmap="RdYlGn", vmin=-lim, vmax=lim)
ax.set_xticks(range(len(CLASS_NAMES)), [f"{c}\n(n={int((test_df['label_name'] == c).sum())})" for c in CLASS_NAMES])
ax.set_yticks(range(len(ap_delta)), ap_delta.index)
for i in range(ap_delta.shape[0]):
    for j in range(ap_delta.shape[1]):
        ax.text(j, i, f"{ap_table.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("AP por clase en test (texto) y diferencia con el baseline (color)")
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de validación de todos los experimentos
fig, ax = plt.subplots(figsize=(10, 5))
for exp in EXPERIMENTS:
    hist = json.loads((EXPERIMENTS_DIR / exp["name"] / "history.json").read_text())
    ax.plot(range(1, len(hist["valid_map"]) + 1), hist["valid_map"], marker="o", markersize=3, label=exp["name"])
ax.set_xlabel("Época"), ax.set_ylabel("mAP valid"), ax.legend(fontsize=8)
plt.show()

## 5. Cómo interpretar los resultados

- **Diferencias pequeñas (< ~0.01 de mAP) pueden ser ruido**: cambiar solo la semilla mueve el resultado. Si dos
  experimentos quedan muy cerca, repítelos con otra `seed` antes de concluir.
- **Compara cada aumentación con `baseline_repetido`**, no solo con `baseline`: así separas el efecto de la
  aumentación del de entrenar 3× más iteraciones.
- **Elige la aumentación con el mAP de validación** y usa el test solo para confirmar: si eliges mirando el test, el
  test deja de ser una estimación honesta.
- Mira también **AP por clase** y **día / noche**: una aumentación puede no cambiar el mAP global (dominado por los
  cérvidos) y, aun así, mejorar mucho las especies raras o las fotos nocturnas.
- Una vez elegida la mejor, puedes reentrenar con el modelo grande (`02_train.ipynb` con
  `train_set = det.DonanaDataset(train_df, IMAGES_DIR, transform=det.get_augmentation("..."), num_augments=2)`) y
  evaluarlo con `03_test.ipynb`.